# **read data**

In [ ]:
import numpy as np
import pandas as pd
import sys

In [ ]:
data = pd.read_csv("/content/Result_17.csv")

In [ ]:
data.head(2)

,YEAR,WEEK,TIMERANGE,LEG,DEPTIME,ARRTIME,DEPDATE,LF
0,2026,25,04 - 06,DADSGN,545,720,2026-06-16,5
1,2026,29,20 - 22,SGNHUI,2110,2240,2026-07-15,0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152277 entries, 0 to 152276
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   YEAR       152277 non-null  int64 
 1   WEEK       152277 non-null  int64 
 2   TIMERANGE  152277 non-null  object
 3   LEG        152277 non-null  object
 4   DEPTIME    152277 non-null  int64 
 5   ARRTIME    152277 non-null  int64 
 6   DEPDATE    152277 non-null  object
 7   LF         152277 non-null  int64 
dtypes: int64(5), object(3)
memory usage: 9.3+ MB


# **function**

In [ ]:
# ham chuyen doi hhmm sang tong so phut
def time_to_minutes(time):
    hours = int(time[:2])
    minutes = int(time[2:])
    return hours * 60 + minutes

In [ ]:
# ham chuyen so phut sang format hhmm
def minutes_to_time(minute):
    hours = int(minute // 60)
    minutes = int(minute % 60)
    return f"{hours:02d}{minutes:02d}"

# **ham nhan input tu nguoi dung**

In [ ]:
def input_info(data):
  print("Type 'quit' or 'exit' at any prompt to stop the program.")

  # nhan thong tin year, leg, loai hanh trinh tu nguoi dung, bao loi va dung chuong trinh neu thong tin nhap vao khong hop le
  # nhap thong tin year
  year_str = input("Enter year: ")
  if year_str.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None
  try:
    year = int(year_str)
  except ValueError:
    print(f"Error: Year '{year_str}' is not a valid number.")
    return None

  # nhap thong tin loai hanh trinh, chi chap nhan 2 gia tri inbound/outbound
  while True:
    leg_type = input("Enter leg type (inbound/outbound): ")
    if leg_type.lower() in ['quit', 'exit']:
      print("Exiting program.")
      return None
    if leg_type.lower() in ['inbound', 'outbound']:
      break
    else:
      print("Leg type must be 'inbound' or 'outbound'. Please try again.")

  # nhap thong tin leg
  leg = input("Enter leg: ")
  if leg.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None

  # print thong tin khung gio cua leg, year vua nhap
  time_range_options = data[(data['YEAR'] == year) & (data['LEG'] == leg)]['TIMERANGE'].unique()
  if len(time_range_options) == 0:
    print(f"Error: No data found for Leg '{leg}', Year {year}.")
    return None
  print(f"Time range for leg '{leg}' of year {year}: {np.sort(time_range_options)}")

  # nhap thong tin time range
  time_range_str = input("Enter time range: ")
  if time_range_str.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None
  time_range = time_range_str

  # filter the data and return the required columns as a DataFrame
  filtered_data = data[(data['YEAR'] == year) & (data['LEG'] == leg) & (data['TIMERANGE'] == time_range)]
  # raise error if filtered_data empty
  if filtered_data.empty:
    print(f"Error: No data found for Leg '{leg}', Year {year}, Time Range '{time_range}'.")
    return None

  # select and return the desired columns, using existing 'LF'
  return_df = filtered_data[['YEAR', 'WEEK', 'LEG', 'DEPTIME', 'ARRTIME', 'TIMERANGE', 'LF']].copy()

  # avg arr/dep time based on leg_type
  if leg_type.lower() == 'inbound':
    return_df['Time_Str'] = return_df['ARRTIME'].astype(str).str.zfill(4)
    return_df['Time_Mean'] = return_df['Time_Str'].apply(time_to_minutes)
    time_column_name_for_output = 'AvgArrTime'
  else:
    return_df['Time_Str'] = return_df['DEPTIME'].astype(str).str.zfill(4)
    return_df['Time_Mean'] = return_df['Time_Str'].apply(time_to_minutes)
    time_column_name_for_output = 'AvgDepTime'

  # groupby and calculate mean for LF and Avg_Minutes
  result = return_df.groupby(['YEAR', 'WEEK', 'LEG', 'TIMERANGE']).agg(LF = ('LF', 'mean'), Avg_Minutes = ('Time_Mean', 'mean')).reset_index()

  # Round LF to the nearest integer
  result['LF'] = result['LF'].round(0).astype(int)

  # tinh avg time
  result[time_column_name_for_output] = result['Avg_Minutes'].apply(minutes_to_time)

  # chon cot hien thi o output
  input_df = result[['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', time_column_name_for_output]]
  #print(input_df)
  return input_df, leg_type

# **ham thuc thi neu nguoi dung chon loai hanh trinh inbound**

In [ ]:
def inbound(leg_input, year, time_range):
  target_arrival_airport = leg_input[-3:]

  original_leg_data = data[(data['YEAR'] == year) & (data['LEG'] == leg_input) & (data['TIMERANGE'] == time_range)].copy()

  if original_leg_data.empty:
      return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'DEPTIME'])

  # 1. Prepare original_leg_data for date/time calculations
  original_leg_data['DEPDATE_dt'] = pd.to_datetime(original_leg_data['DEPDATE'])
  original_leg_data['DEPTIME_val'] = original_leg_data['DEPTIME']
  original_leg_data['ARRTIME_val'] = original_leg_data['ARRTIME']

  # Calculate ARRDATE for each flight in original_leg_data (considering overnight flights)
  original_leg_data['CALCULATED_ARRDATE_dt'] = original_leg_data.apply(
      lambda row: row['DEPDATE_dt'] + pd.Timedelta(days=1) if row['ARRTIME_val'] < row['DEPTIME_val'] else row['DEPDATE_dt'],
      axis=1
  )

  # Calculate average arrival time (in minutes) for original leg per week
  original_leg_data['Time_Str'] = original_leg_data['ARRTIME_val'].astype(str).str.zfill(4)
  original_leg_data['Time_Min'] = original_leg_data['Time_Str'].apply(time_to_minutes)
  original_leg_weekly_arrival_time = original_leg_data.groupby(['YEAR', 'WEEK'])['Time_Min'].mean().reset_index()
  original_leg_weekly_arrival_time = original_leg_weekly_arrival_time.rename(columns={'Time_Min': 'original_leg_avg_arrival_time'})

  # Determine a representative ARRDATE for original leg per week (e.g., the min date, assuming coherence within a week)
  original_leg_weekly_arrdate = original_leg_data.groupby(['YEAR', 'WEEK'])['CALCULATED_ARRDATE_dt'].min().reset_index()
  original_leg_weekly_arrdate = original_leg_weekly_arrdate.rename(columns={'CALCULATED_ARRDATE_dt': 'original_leg_representative_arrdate'})


  # Filter context data for potential outbound legs
  filtered_context_data = data[(data['YEAR'] == year)].copy()
  potential_outbound_legs = filtered_context_data[filtered_context_data['LEG'].str.startswith(target_arrival_airport)].copy()

  if potential_outbound_legs.empty:
      return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'DEPTIME'])

  # 2. Prepare potential_outbound_legs for date/time calculations
  potential_outbound_legs['DEPDATE_dt'] = pd.to_datetime(potential_outbound_legs['DEPDATE'])
  potential_outbound_legs['DEPTIME_val'] = potential_outbound_legs['DEPTIME']
  potential_outbound_legs['DEPTIME_Str'] = potential_outbound_legs['DEPTIME_val'].astype(str).str.zfill(4)
  potential_outbound_legs['DEPTIME_Min'] = potential_outbound_legs['DEPTIME_Str'].apply(time_to_minutes)

  # 3. Merge and apply filtering conditions
  # Merge original leg's representative arrival date and average arrival time (in minutes)
  merged_outbound = pd.merge(potential_outbound_legs, original_leg_weekly_arrdate, on=['YEAR', 'WEEK'], how='left')
  merged_outbound = pd.merge(merged_outbound, original_leg_weekly_arrival_time, on=['YEAR', 'WEEK'], how='left')


  # Define conditions
  # Condition 1: depdate = arrdate of leg input AND deptime > arrtime of leg input
  condition1 = (merged_outbound['DEPDATE_dt'] == merged_outbound['original_leg_representative_arrdate']) & \
               (merged_outbound['DEPTIME_Min'] > merged_outbound['original_leg_avg_arrival_time'])

  # Condition 2: depdate = arrdate of leg input + 1 day AND deptime < arrtime of leg input
  condition2 = (merged_outbound['DEPDATE_dt'] == (merged_outbound['original_leg_representative_arrdate'] + pd.Timedelta(days=1))) & \
               (merged_outbound['DEPTIME_Min'] < merged_outbound['original_leg_avg_arrival_time'])

  # Apply the combined filter
  filtered_by_date_time = merged_outbound[condition1 | condition2].copy()

  if filtered_by_date_time.empty:
      return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'DEPTIME'])

  # Continue with grouping and calculations on filtered_by_date_time
  # group by leg, time range, WEEK de tinh LF, avg deptime
  grouped_outbound = filtered_by_date_time.groupby(['YEAR', 'WEEK', 'LEG', 'TIMERANGE']).agg(
      LF=('LF', 'mean'),
      avg_deptime_min=('DEPTIME_Min', 'mean')
  ).reset_index()

  # Round LF to the nearest integer
  grouped_outbound['LF'] = grouped_outbound['LF'].round(0).astype(int)

  # tinh stopovertime = avg deptime - avg arrival time (now merged by week)
  # We need to re-merge original_leg_weekly_arrival_time to calculate STOPOVERTIME for the grouped data
  grouped_outbound = pd.merge(grouped_outbound, original_leg_weekly_arrival_time, on=['YEAR', 'WEEK'], how='left')
  grouped_outbound['STOPOVERTIME_in_minutes'] = grouped_outbound['avg_deptime_min'] - grouped_outbound['original_leg_avg_arrival_time']
  grouped_outbound['STOPOVERTIME_in_minutes'] = grouped_outbound['STOPOVERTIME_in_minutes'].apply(lambda x: x + 24 * 60 if x < 0 else x)
  grouped_outbound['STOPOVERTIME_in_minutes'] = grouped_outbound['STOPOVERTIME_in_minutes'].fillna(0) # fill NaN with 0
  grouped_outbound['STOPOVERTIME'] = grouped_outbound['STOPOVERTIME_in_minutes'].apply(minutes_to_time)

  # quy doi avg deptime ve format hhmm
  grouped_outbound['DEPTIME'] = grouped_outbound['avg_deptime_min'].apply(minutes_to_time)

  # output
  result_df = grouped_outbound[['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'DEPTIME']].copy()
  result_df = result_df.sort_values(by = ['YEAR', 'WEEK', 'LEG', 'TIMERANGE'], ascending = True).reset_index(drop = True)

  return result_df

# **ham thuc thi neu nguoi dung chon loai hanh trinh outbound**

In [ ]:
def outbound(leg_input, year, time_range):
  target_departure_airport = leg_input[:3]

  original_leg_data = data[(data['YEAR'] == year) & (data['LEG'] == leg_input) & (data['TIMERANGE'] == time_range)].copy()

  if original_leg_data.empty:
      return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'ARRTIME'])

  original_leg_data['DEPDATE_dt'] = pd.to_datetime(original_leg_data['DEPDATE'])
  original_leg_data['DEPTIME_val'] = original_leg_data['DEPTIME']
  original_leg_data['DEPTIME_Str'] = original_leg_data['DEPTIME_val'].astype(str).str.zfill(4)
  original_leg_data['DEPTIME_Min'] = original_leg_data['DEPTIME_Str'].apply(time_to_minutes)

  # Calculate weekly average DEPTIME and representative DEPDATE for the original outbound leg
  original_leg_weekly_deptime = original_leg_data.groupby(['YEAR', 'WEEK'])['DEPTIME_Min'].mean().reset_index()
  original_leg_weekly_deptime = original_leg_weekly_deptime.rename(columns={'DEPTIME_Min': 'original_leg_avg_deptime'})

  original_leg_weekly_depdate = original_leg_data.groupby(['YEAR', 'WEEK'])['DEPDATE_dt'].min().reset_index()
  original_leg_weekly_depdate = original_leg_weekly_depdate.rename(columns={'DEPDATE_dt': 'original_leg_representative_depdate'})

  potential_inbound_legs = data[(data['YEAR'] == year) & (data['LEG'].str.endswith(target_departure_airport))].copy()

  if potential_inbound_legs.empty:
      return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'ARRTIME'])

  potential_inbound_legs['DEPDATE_dt'] = pd.to_datetime(potential_inbound_legs['DEPDATE'])
  potential_inbound_legs['ARRTIME_val'] = potential_inbound_legs['ARRTIME']
  potential_inbound_legs['DEPTIME_val'] = potential_inbound_legs['DEPTIME'] # Needed for calculating CALCULATED_ARRDATE_dt for inbound leg

  # Calculate arrival date of the inbound leg (CALCULATED_ARRDATE_dt) based on its own DEPDATE and ARRTIME
  potential_inbound_legs['CALCULATED_ARRDATE_dt'] = potential_inbound_legs.apply(
      lambda row: row['DEPDATE_dt'] + pd.Timedelta(days=1) if row['ARRTIME_val'] < row['DEPTIME_val'] else row['DEPDATE_dt'],
      axis=1
  )
  potential_inbound_legs['ARRTIME_Str'] = potential_inbound_legs['ARRTIME_val'].astype(str).str.zfill(4)
  potential_inbound_legs['ARRTIME_Min'] = potential_inbound_legs['ARRTIME_Str'].apply(time_to_minutes)

  # Merge original leg's representative depdate and average deptime (in minutes)
  merged_inbound = pd.merge(potential_inbound_legs, original_leg_weekly_depdate, on=['YEAR', 'WEEK'], how='left')
  merged_inbound = pd.merge(merged_inbound, original_leg_weekly_deptime, on=['YEAR', 'WEEK'], how='left')

  # Drop rows where merge failed (no corresponding original leg data for the week)
  merged_inbound.dropna(subset=['original_leg_representative_depdate', 'original_leg_avg_deptime'], inplace=True)
  if merged_inbound.empty:
      return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'ARRTIME'])

  # Define a function to calculate STOPOVERTIME based on the specified conditions
  def calculate_stopovertime_new_logic(row):
      original_dep_date = row['original_leg_representative_depdate']
      original_avg_deptime = row['original_leg_avg_deptime']
      inbound_arr_date = row['CALCULATED_ARRDATE_dt']
      inbound_arr_time_min = row['ARRTIME_Min']

      # Case 1: arrdate of inbound leg = depdate of input leg - 1 and arrtime of inbound > deptime of input
      if inbound_arr_date == (original_dep_date - pd.Timedelta(days=1)) and inbound_arr_time_min > original_avg_deptime:
          return (original_avg_deptime - inbound_arr_time_min) + (24 * 60)
      # Case 2: arrdate of inbound leg = depdate of input leg and arrtime of inbound < deptime of input
      elif inbound_arr_date == original_dep_date and inbound_arr_time_min < original_avg_deptime:
          return original_avg_deptime - inbound_arr_time_min
      else:
          return np.nan # If conditions are not met, return NaN

  merged_inbound['STOPOVERTIME_in_minutes'] = merged_inbound.apply(calculate_stopovertime_new_logic, axis=1)

  # Filter out rows where STOPOVERTIME could not be calculated (NaN)
  filtered_by_date_time = merged_inbound.dropna(subset=['STOPOVERTIME_in_minutes']).copy()

  if filtered_by_date_time.empty:
      return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'ARRTIME'])

  # Group by leg, time range, WEEK to calculate LF and avg ARRTME, and avg STOPOVERTIME
  grouped_inbound = filtered_by_date_time.groupby(['YEAR', 'WEEK', 'LEG', 'TIMERANGE']).agg(
      LF=('LF', 'mean'),
      avg_arrtime_min=('ARRTIME_Min', 'mean'),
      STOPOVERTIME_in_minutes=('STOPOVERTIME_in_minutes', 'mean')
  ).reset_index()

  # Round LF to the nearest integer
  grouped_inbound['LF'] = grouped_inbound['LF'].round(0).astype(int)

  # Convert average STOPOVERTIME and ARRTIME to hhmm format
  grouped_inbound['STOPOVERTIME'] = grouped_inbound['STOPOVERTIME_in_minutes'].apply(minutes_to_time)
  grouped_inbound['ARRTIME'] = grouped_inbound['avg_arrtime_min'].apply(minutes_to_time)

  # output
  result_df = grouped_inbound[['YEAR', 'WEEK', 'LEG', 'TIMERANGE', 'LF', 'STOPOVERTIME', 'ARRTIME']].copy()
  result_df = result_df.sort_values(by=['YEAR', 'WEEK', 'LEG', 'TIMERANGE'], ascending=True).reset_index(drop=True)

  return result_df

# **output**

In [ ]:
# goi ham input_info khi nguoi dung nhap thong tin
user_inputs = input_info(data)

if user_inputs is not None:
  df_input_info, leg_type_input = user_inputs
  leg_input = df_input_info['LEG'].iloc[0]
  year_input = df_input_info['YEAR'].iloc[0]
  time_range_input = df_input_info['TIMERANGE'].iloc[0]

  print("User Input Details:")
  display(df_input_info)

  # goi ham inbound/outbound tuy theo loai hanh trinh nguoi dung nhap
  if leg_type_input.lower() == 'inbound':
    result_df = inbound(leg_input, year_input, time_range_input)
    if not result_df.empty:
      print("\nInbound Legs:")
      display(result_df)
    else:
      print("No inbound legs found for the given input.")
  elif leg_type_input.lower() == 'outbound':
    result_df = outbound(leg_input, year_input, time_range_input)
    if not result_df.empty:
      print("\nOutbound Legs:")
      display(result_df)
    else:
      print("No outbound legs found for the given input.")
else:
  print("Input process terminated or invalid input provided. No further processing in this cell.")

Type 'quit' or 'exit' at any prompt to stop the program.
Enter year: 2026
Enter leg type (inbound/outbound): outbound
Enter leg: HANBKK
Time range for leg 'HANBKK' of year 2026: ['08 - 10' '12 - 14' '14 - 16' '16 - 18' '18 - 20']
Enter time range: 12 - 14
User Input Details:


,YEAR,WEEK,LEG,TIMERANGE,LF,AvgDepTime
0,2026,11,HANBKK,12 - 14,88,1240
1,2026,12,HANBKK,12 - 14,82,1240
2,2026,13,HANBKK,12 - 14,73,1240
3,2026,14,HANBKK,12 - 14,62,1245
4,2026,15,HANBKK,12 - 14,69,1245
5,2026,16,HANBKK,12 - 14,68,1245
6,2026,17,HANBKK,12 - 14,62,1245
7,2026,18,HANBKK,12 - 14,52,1245
8,2026,19,HANBKK,12 - 14,37,1245
9,2026,20,HANBKK,12 - 14,44,1245



Outbound Legs:


,YEAR,WEEK,LEG,TIMERANGE,LF,STOPOVERTIME,ARRTIME
0,2026,11,BMVHAN,08 - 10,70,0125,1115
1,2026,11,CEBHAN,04 - 06,68,0505,0735
2,2026,11,DADHAN,06 - 08,64,0415,0825
3,2026,11,DADHAN,08 - 10,99,0125,1115
4,2026,11,DADHAN,10 - 12,81,0055,1145
...,...,...,...,...,...,...,...
973,2026,53,TPEHAN,06 - 08,0,0250,0950
974,2026,53,VCAHAN,08 - 10,0,0050,1150
975,2026,53,VCLHAN,08 - 10,0,0200,1040
976,2026,53,VDHHAN,10 - 12,0,0110,1130


In [ ]:
df_input_info = df_input_info.pivot(index = ['YEAR', 'LEG', 'TIMERANGE'], columns = 'WEEK', values = ['LF']).reset_index()
df_input_info

YEAR     LEG TIMERANGE  LF                          ...                 \
WEEK                          11  12  13  14  15  16  17  ... 44 45 46 47 48   
0     2026  HANBKK   12 - 14  88  82  73  62  69  68  62  ...  2  3  2  3  2   

                     
WEEK 49 50 51 52 53  
0     2  2  1  2  2  

[1 rows x 46 columns]

In [ ]:
# export df_input_info ra file csv
df_input_info.to_csv('input_info.csv', index = False)
# download file csv
import os
full_path = os.path.abspath('input_info.csv')
print(f"File path: {full_path}")

File path: c:\Users\hiennguyenthithanh\Downloads\input_info.csv


In [ ]:
result_df = result_df.pivot(index = ['LEG', 'TIMERANGE', 'ARRTIME', 'STOPOVERTIME'], columns = 'WEEK', values = ['LF']).reset_index()
result_df

LEG TIMERANGE ARRTIME STOPOVERTIME    LF                          \
WEEK                                           11    12    13    14    15   
0     BMVHAN   08 - 10    1105         0139   NaN   NaN   NaN   NaN   NaN   
1     BMVHAN   08 - 10    1105         0140   NaN   NaN   NaN  61.0  12.0   
2     BMVHAN   08 - 10    1115         0125  70.0  82.0  61.0   NaN   NaN   
3     BMVHAN   08 - 10    1120         0120   NaN   NaN   NaN   NaN   NaN   
4     CEBHAN   04 - 06    0735         0505  68.0  73.0  65.0   NaN   NaN   
..       ...       ...     ...          ...   ...   ...   ...   ...   ...   
84    VIIHAN   08 - 10    0915         0329   NaN   NaN   NaN   NaN   NaN   
85    VIIHAN   08 - 10    0915         0330   NaN   NaN   NaN   NaN  10.0   
86    VIIHAN   08 - 10    0930         0310   NaN   NaN   NaN   NaN   NaN   
87    VIIHAN   08 - 10    1005         0235  54.0  73.0  11.0   NaN   NaN   
88    VIIHAN   08 - 10    1005         0240   NaN   NaN   NaN  16.0   NaN   

            ...                                                    
WEEK    16  ...   44   45   46   47   48   49   50   51   52   53  
0      NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
1     16.0  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
2      NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
3      NaN  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
4      NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
..     ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  
84     NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
85     5.0  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
86     NaN  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
87     NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
88     NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[89 rows x 47 columns]

In [ ]:
# export result_df ra file csv
result_df.to_csv('result_df.csv', index = False)
# download file csv
import os
full_path = os.path.abspath('result_df.csv')
print(f"File path: {full_path}")

File path: c:\Users\hiennguyenthithanh\Downloads\result_df.csv
